# 포트홀 탐지 YOLO 파인튜닝 (Colab, 무료 GPU)

도로 손상(포트홀) 탐지 모델을 파인튜닝하고 **전/후 성능(mAP)**을 비교합니다.
제안서 ⑥ 단계의 핵심 결과물(파인튜닝 전후 성능표)을 만듭니다.

**먼저 GPU 켜기:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → 하드웨어 가속기 = **GPU (T4)**

In [ ]:
!pip -q install ultralytics
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())

## 1. 데이터셋 준비
로컬의 Roboflow Pothole zip(YOLOv8 형식, 예: `Pothole.v1-raw.yolov8.zip`)을 업로드하세요.

In [ ]:
import zipfile, os, glob
from google.colab import files
up = files.upload()  # Pothole zip 선택
zip_name = list(up.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('dataset')
yaml_path = glob.glob('dataset/**/data.yaml', recursive=True)[0]
print('data.yaml:', yaml_path)
print(open(yaml_path).read())

In [ ]:
# data.yaml 경로 정규화 (Colab 절대경로로)
import yaml, os
root = os.path.dirname(yaml_path)
d = yaml.safe_load(open(yaml_path))
d['path'] = os.path.abspath(root)
for split, key in [('train', 'train'), ('valid', 'val'), ('test', 'test')]:
    if os.path.isdir(os.path.join(root, split, 'images')):
        d[key] = f'{split}/images'
open(yaml_path, 'w').write(yaml.safe_dump(d))
print(d)

## 2. 파인튜닝 전 (baseline)
COCO 사전학습 모델에는 'pothole' 클래스가 없어 포트홀을 사실상 못 잡습니다(전 ≈ 0).
아래 학습 후의 mAP와 대비됩니다.

In [ ]:
from ultralytics import YOLO
# 경량 모델(T4에서 빠름). 더 정확히는 yolov8s/m 사용.
model = YOLO('yolov8n.pt')
results = model.train(data=yaml_path, epochs=50, imgsz=640, batch=16, name='pothole_ft')

## 3. 파인튜닝 후 성능 (mAP)

In [ ]:
metrics = model.val(data=yaml_path, split='test')
print('mAP@0.5      :', round(float(metrics.box.map50), 4))
print('mAP@0.5:0.95 :', round(float(metrics.box.map), 4))

In [ ]:
# 샘플 예측 시각화
import glob, matplotlib.pyplot as plt
test_imgs = glob.glob(os.path.join(d['path'], 'test', 'images', '*'))[:4]
for r in model(test_imgs, verbose=False):
    plt.figure(figsize=(6, 4)); plt.imshow(r.plot()[..., ::-1]); plt.axis('off'); plt.show()

## 4. 파인튜닝 모델 내려받기
`best.pt`를 다운로드해 프로젝트의 `models/pothole_yolo.pt` 로 저장하세요.

그러면 로컬에서:
```
python -m vlm detect <img> --detector yolo
python -m vlm compare data/images/real --backends gemini,yolo   # 파인튜닝 YOLO vs Gemini
python -m vlm label data/images/real --detector yolo            # 파인튜닝 모델로 라벨 생성
```

In [ ]:
from google.colab import files
best = 'runs/detect/pothole_ft/weights/best.pt'
print(best, round(os.path.getsize(best) / 1e6, 1), 'MB')
files.download(best)